# Benchmark Analysis and Time-vs-Memory

This notebook analyzes one benchmark run at a time using the stable database schema. The goal is to compare latency, throughput, and memory behavior without any schema guessing or notebook-specific column heuristics.

The connection bootstrap is environment-aware. It will try `BENCH_DB_URL` first, then a local Postgres URL on `localhost`, and then the Docker Compose service hostname `db`. That makes the notebook usable both from a local editor session and from the profiler or performance-monitoring service container.

## Stage Definitions

- **Stage 1: Serialization / transformation**
  Measure the cost of converting native in-memory clinical data into a wire-ready representation. Timing starts immediately before the first field write and stops as soon as the payload is sealed.
- **Stage 2: Transport / wire movement**
  Measure the cost of moving already-serialized bytes across the transport path. Timing starts at the send call and ends at transport completion.
- **Stage 3: Query / traversal**
  Measure the cost of answering a clinical question directly from the received representation. Timing starts at the first parser or reader call and ends when the target value is extracted.
- **Stage 3 materialize**
  Measure the cost of eagerly reconstructing native structs from the received representation when downstream code requires materialized objects.

The current local implementation is expected to populate Stage 1 and Stage 3 query for the `fastfhir` and `json_fhir` arms.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
plt.style.use('ggplot')


def build_db_url(host):
    user = os.getenv('POSTGRES_USER', 'bench')
    password = os.getenv('POSTGRES_PASSWORD', 'bench')
    database = os.getenv('POSTGRES_DB', 'benchmark')
    port = os.getenv('POSTGRES_PORT', '5432')
    return f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'


def candidate_db_urls():
    configured = os.getenv('BENCH_DB_URL', '').strip()
    candidates = [
        configured,
        build_db_url('localhost'),
        build_db_url('db'),
    ]

    seen = set()
    ordered = []
    for url in candidates:
        if url and url not in seen:
            ordered.append(url)
            seen.add(url)
    return ordered


def connect_engine():
    errors = []
    for url in candidate_db_urls():
        engine = create_engine(url, future=True, pool_pre_ping=True)
        try:
            with engine.connect() as conn:
                conn.execute(text('SELECT 1'))
            return engine, url
        except Exception as exc:
            errors.append(f'{url} -> {exc}')
            engine.dispose()

    raise RuntimeError('Unable to connect to benchmark database. Tried:\n' + '\n'.join(errors))


def query_frame(sql, params=None):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params or {})


engine, DB_URL = connect_engine()
print('Using DB URL:', DB_URL)

In [ ]:
runs_df = query_frame(
    """
    SELECT run_id, environment_name, created_at, last_metric_ts, metric_rows, aggregate_rows
    FROM v_latest_run_status
    ORDER BY created_at DESC, run_id DESC
    LIMIT 20
    """
)

display(runs_df)

SELECTED_RUN_ID = os.getenv('BENCH_RUN_ID', '').strip()
if not SELECTED_RUN_ID:
    if runs_df.empty:
        raise RuntimeError('No benchmark runs were found in v_latest_run_status.')
    SELECTED_RUN_ID = runs_df.iloc[0]['run_id']

print('Selected run:', SELECTED_RUN_ID)

In [ ]:
agg_df = query_frame(
    """
    SELECT run_id, arm, stage, n_samples, p50_ms, p95_ms, p99_ms, throughput_rps, peak_rss_mb
    FROM aggregate_metrics_table
    WHERE run_id = :run_id
    ORDER BY arm, stage
    """,
    {'run_id': SELECTED_RUN_ID},
)

raw_df = query_frame(
    """
    SELECT run_id, arm, stage, duration_us, start_ts, end_ts, throughput_rps, peak_rss_mb
    FROM raw_metrics_table
    WHERE run_id = :run_id
    ORDER BY arm, stage, start_ts
    """,
    {'run_id': SELECTED_RUN_ID},
)

frontier_df = query_frame(
    """
    SELECT run_id, arm, stage, query_p50_ms, query_p95_ms, query_p99_ms, peak_rss_mb, avg_rss_delta_mb
    FROM v_time_memory_frontier
    WHERE run_id = :run_id
    ORDER BY stage, arm
    """,
    {'run_id': SELECTED_RUN_ID},
)

print('Aggregate rows for selected run:', len(agg_df))
print('Raw rows for selected run:', len(raw_df))
print('Frontier rows for selected run:', len(frontier_df))

display(agg_df)

In [ ]:
stage_order = [
    'stage1_serialize',
    'stage2_transport',
    'stage3_query',
    'stage3_materialize',
]

stage_labels = {
    'stage1_serialize': 'Stage 1 Serialize',
    'stage2_transport': 'Stage 2 Transport',
    'stage3_query': 'Stage 3 Query',
    'stage3_materialize': 'Stage 3 Materialize',
}

latency_summary_df = agg_df[['arm', 'stage', 'n_samples', 'p50_ms', 'p95_ms', 'p99_ms']].copy()
latency_summary_df['stage'] = pd.Categorical(latency_summary_df['stage'], categories=stage_order, ordered=True)
latency_summary_df = latency_summary_df.sort_values(by=['stage', 'arm'])

display(latency_summary_df)

latency_plot_df = raw_df[['arm', 'stage', 'duration_us']].dropna(subset=['duration_us']).copy()
latency_plot_df['stage'] = pd.Categorical(latency_plot_df['stage'], categories=stage_order, ordered=True)
latency_plot_df = latency_plot_df.sort_values(by=['stage', 'arm'])
latency_plot_df['duration_ms'] = pd.to_numeric(latency_plot_df['duration_us'], errors='coerce') / 1000.0
latency_plot_df = latency_plot_df.dropna(subset=['duration_ms'])
latency_plot_df['series_label'] = latency_plot_df.apply(
    lambda row: f"{stage_labels.get(row['stage'], row['stage'])}\n{row['arm']}",
    axis=1,
)

if latency_plot_df.empty:
    print('No raw latency rows are available for the selected run.')
else:
    ordered_labels = []
    for stage in stage_order:
        stage_rows = latency_plot_df[latency_plot_df['stage'] == stage]
        for arm in list(stage_rows['arm'].astype(str).drop_duplicates()):
            ordered_labels.append(f"{stage_labels.get(stage, stage)}\n{arm}")

    boxplot_data = []
    for label in ordered_labels:
        selected_frame = latency_plot_df.loc[
            latency_plot_df['series_label'] == label,
            ['duration_ms'],
        ]
        numeric_values = pd.to_numeric(selected_frame['duration_ms'], errors='coerce').dropna()
        boxplot_data.append(numeric_values.to_numpy(dtype=float).tolist())

    sample_counts = [len(values) for values in boxplot_data]
    min_samples = min(sample_counts) if sample_counts else 0

    fig, ax = plt.subplots(figsize=(11, 6))

    if min_samples >= 5:
        ax.boxplot(boxplot_data, tick_labels=ordered_labels, patch_artist=True, showfliers=True)
        ax.set_title(f'Stage latency distribution for {SELECTED_RUN_ID}')
    else:
        x_positions = list(range(1, len(ordered_labels) + 1))
        for index, values in enumerate(boxplot_data, start=1):
            ax.scatter([index] * len(values), values, alpha=0.8)

        ax.set_xticks(x_positions)
        ax.set_xticklabels(ordered_labels)
        ax.set_title(f'Stage latency samples for {SELECTED_RUN_ID}')
        print('Box-and-whisker plots become informative once each stage/arm has at least 5 samples. Showing raw sample points for this run.')

    ax.set_xlabel('Stage and arm')
    ax.set_ylabel('Latency (ms)')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
tail_df = agg_df[['arm', 'stage', 'n_samples', 'p50_ms', 'p95_ms', 'p99_ms', 'throughput_rps']].copy()
tail_df['stage'] = pd.Categorical(tail_df['stage'], categories=stage_order, ordered=True)
tail_df = tail_df.sort_values(by=['stage', 'arm'])

display(tail_df)

throughput_source_df = tail_df[['arm', 'throughput_rps']].dropna(subset=['throughput_rps'])

if throughput_source_df.empty:
    print('No throughput values are available for the selected run yet.')
else:
    throughput_df = (
        throughput_source_df
        .groupby('arm', as_index=False)
        .median(numeric_only=True)
        .sort_values(by='throughput_rps', ascending=False)
    )

    display(throughput_df)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(throughput_df['arm'], throughput_df['throughput_rps'])
    ax.set_title(f'Median throughput by arm for {SELECTED_RUN_ID}')
    ax.set_xlabel('Arm')
    ax.set_ylabel('Throughput (rows/sec)')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

In [ ]:
display(frontier_df)

frontier_plot_df = frontier_df.dropna(subset=['query_p50_ms', 'peak_rss_mb']).copy()
frontier_plot_df['stage_label'] = frontier_plot_df['stage'].map(stage_labels).fillna(frontier_plot_df['stage'])

if frontier_plot_df.empty:
    print('No time-vs-memory frontier points are available for the selected run.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(frontier_plot_df['peak_rss_mb'], frontier_plot_df['query_p50_ms'])

    for _, row in frontier_plot_df.iterrows():
        ax.annotate(
            f"{row['arm']} | {row['stage_label']}",
            (row['peak_rss_mb'], row['query_p50_ms']),
            xytext=(5, 5),
            textcoords='offset points',
        )

    ax.set_title(f'Time-vs-memory frontier for {SELECTED_RUN_ID}')
    ax.set_xlabel('Peak RSS (MB)')
    ax.set_ylabel('P50 latency (ms)')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
run_persistence_df = query_frame(
    """
    SELECT
        run_id,
        COUNT(*) AS raw_rows,
        COUNT(*) FILTER (WHERE peak_rss_mb IS NOT NULL) AS rows_with_peak_rss,
        COUNT(*) FILTER (WHERE throughput_rps IS NOT NULL) AS rows_with_throughput
    FROM raw_metrics_table
    WHERE run_id = :run_id
    GROUP BY run_id
    """,
    {'run_id': SELECTED_RUN_ID},
)

display(run_persistence_df)
print('Generated analysis artifacts are not written to disk. The source of truth is the benchmark database for the selected run.')

## Interpretation Notes

Read the plots with the stage boundaries in mind:

1. Stage 1 answers how expensive it is to create the wire-ready representation.
2. Stage 2 answers how expensive it is to move bytes once the payload already exists.
3. Stage 3 query answers how expensive it is to extract a clinically relevant value directly from the received representation.
4. Stage 3 materialize answers how expensive it is to rebuild native structs when traversal alone is not sufficient.

For the current local smoke runs, the most important comparison is usually `fastfhir` versus `json_fhir` on Stage 1 and Stage 3 query. If Stage 2 or Stage 3 materialize are absent, that indicates those paths have not been instrumented or persisted yet rather than a plotting problem.

## Real-Data Smoke Run (Direct Binary Execution)

This section executes the current benchmark binaries directly and parses the emitted metric lines (`arm,stage,duration_us`) from stdout.

Use this when validating a code/data-path change before database persistence jobs are run.

In [ ]:
import re
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / 'notebooks').exists():
    pass
elif (repo_root.parent / 'notebooks').exists():
    repo_root = repo_root.parent
else:
    raise RuntimeError(f'Could not locate repository root from {Path.cwd()}')

cmd = (
    'cmake --build build/bench '
    '&& ./build/bench/bench/bench_harness --smoke --iterations 1 '
    '&& ./build/bench/bench/bench_timing_conformance'
)

completed = subprocess.run(
    cmd,
    cwd=repo_root,
    shell=True,
    check=False,
    text=True,
    capture_output=True,
)

combined_output = (completed.stdout or '') + '\n' + (completed.stderr or '')
print(combined_output)
print(f'Exit code: {completed.returncode}')

metric_pattern = re.compile(r'^(?P<arm>[^,]+),(?P<stage>[^,]+),(?P<duration_us>\d+)$')
rows = []
for line in combined_output.splitlines():
    match = metric_pattern.match(line.strip())
    if match:
        rows.append(
            {
                'arm': match.group('arm'),
                'stage': match.group('stage'),
                'duration_us': int(match.group('duration_us')),
            }
        )

if not rows:
    raise RuntimeError('No metric lines were found in command output.')

latest_run_df = pd.DataFrame(rows)
latest_run_df['duration_ms'] = latest_run_df['duration_us'] / 1000.0

display(latest_run_df)

pivot_latest = latest_run_df.pivot(index='stage', columns='arm', values='duration_us').sort_index()
display(pivot_latest)

ax = pivot_latest.plot(kind='bar', figsize=(9, 4.5), rot=20)
ax.set_title('Latest real-data smoke run (duration_us)')
ax.set_xlabel('Stage')
ax.set_ylabel('Duration (microseconds)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()